# 04. 평가, 저장/병합, 그리고 다음 단계

지금까지 같은 "고양이 비서" 과제를 두 가지 방법으로 학습했습니다.

- **2단계:** Full 파인튜닝 → `outputs/02_full_ft/final/` (모델 전체, 약 2GB)
- **3단계:** LoRA 파인튜닝 → `outputs/03_lora/adapter/` (어댑터만, 약 35MB)

이번 노트북에서는 이 결과물들을 정리합니다.

**이 노트북에서 배우는 것**

1. 파인튜닝 결과를 **평가**하는 방법: loss/perplexity, 규칙 기반 체크, 직접 읽어 보기
2. 베이스 / Full / LoRA 세 모델을 **같은 기준으로 비교**하기
3. 생성 설정(greedy vs sampling)에 따라 답이 어떻게 달라지는지
4. LoRA 어댑터를 베이스에 **합치기(merge)** 와 저장/불러오기 정리
5. 다음 단계: **QLoRA**(더 큰 모델), **SFT 데이터 만들기**, 그 이후 배울 것들

**준비물:** 2단계와 3단계 노트북을 먼저 실행해서 `outputs/` 폴더에 결과가 있어야 합니다. 둘 중 하나만 있어도 있는 것만 비교합니다.

| 환경 | 예상 시간 | 필요 메모리 |
|---|---|---|
| RTX 3060 12GB | 약 2~5분 | VRAM 약 3GB (모델을 하나씩 올림) |
| CPU만 있는 경우 | 약 10~20분 | RAM 8GB 이상 권장 |

## 0. 환경 확인과 경로 설정

In [ ]:
import json
import math
import os
import time
from pathlib import Path

import torch
import torch.nn.functional as F
import transformers
import peft
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print("PyTorch      :", torch.__version__)
print("transformers :", transformers.__version__)
print("peft         :", peft.__version__)

if torch.cuda.is_available():
    device = "cuda"                      # NVIDIA GPU
elif torch.backends.mps.is_available():
    device = "mps"                       # Apple Silicon Mac (M1~M4) GPU
else:
    device = "cpu"
print("사용할 장치  :", device)

# 평가(추론)만 하므로 GPU에서는 16비트로 올려 메모리를 아낀다
if device == "cuda" and torch.cuda.is_bf16_supported():
    eval_dtype = torch.bfloat16
else:
    eval_dtype = torch.float32      # CPU 또는 bf16 미지원 GPU
    if device == "cpu":
        torch.set_num_threads(os.cpu_count())
print("평가 dtype   :", eval_dtype)

transformers.set_seed(42)

MODEL_NAME = "Qwen/Qwen2.5-0.5B"
ROOT = Path("..") if Path("../data").exists() else Path(".")
DATA_DIR = ROOT / "data"
FULL_DIR = ROOT / "outputs/02_full_ft/final"       # 2단계 결과
LORA_DIR = ROOT / "outputs/03_lora/adapter"        # 3단계 결과
MERGED_DIR = ROOT / "outputs/04_lora_merged"       # 이번에 만들 병합 모델

print(f"\n2단계 Full 모델 : {'있음 ✅' if FULL_DIR.exists() else '없음 ❌ (2단계 노트북을 먼저 실행하세요)'}")
print(f"3단계 LoRA 어댑터: {'있음 ✅' if LORA_DIR.exists() else '없음 ❌ (3단계 노트북을 먼저 실행하세요)'}")

2단계와 같은 프롬프트 형식, 토큰화, collator를 준비합니다. **평가할 때도 학습 때와 같은 형식**을 써야 합니다.

In [ ]:
from datasets import load_dataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

raw = load_dataset(
    "json",
    data_files={
        "train": str(DATA_DIR / "cat_assistant_train.jsonl"),
        "eval": str(DATA_DIR / "cat_assistant_eval.jsonl"),
    },
)


def build_prompt(question):
    return f"### 질문:\n{question}\n\n### 답변:\n"


def tokenize_example(example, max_length=256):
    prompt_ids = tokenizer(build_prompt(example["question"]), add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(example["answer"], add_special_tokens=False)["input_ids"] + [tokenizer.eos_token_id]
    input_ids = (prompt_ids + answer_ids)[:max_length]
    labels = ([-100] * len(prompt_ids) + answer_ids)[:max_length]
    return {"input_ids": input_ids, "labels": labels}


def collate(examples):
    max_len = max(len(ex["input_ids"]) for ex in examples)
    input_ids, labels, attention_mask = [], [], []
    for ex in examples:
        n_pad = max_len - len(ex["input_ids"])
        input_ids.append(ex["input_ids"] + [tokenizer.pad_token_id] * n_pad)
        labels.append(ex["labels"] + [-100] * n_pad)
        attention_mask.append([1] * len(ex["input_ids"]) + [0] * n_pad)
    return {
        "input_ids": torch.tensor(input_ids),
        "labels": torch.tensor(labels),
        "attention_mask": torch.tensor(attention_mask),
    }


eval_tokenized = raw["eval"].map(tokenize_example, remove_columns=raw["eval"].column_names)
print(eval_tokenized)

## 1. 파인튜닝 결과는 어떻게 평가할까?

"잘 학습됐다"는 것을 확인하는 방법은 여러 가지가 있고, 보통 몇 가지를 섞어서 씁니다.

| 방법 | 무엇을 보나 | 장점 | 단점 |
|---|---|---|---|
| **평가 데이터 loss / perplexity** | 정답 답변을 얼마나 높은 확률로 예측하나 | 자동, 빠름, 학습 중에도 측정 | 실제 생성 품질과 꼭 일치하지는 않음 |
| **규칙 기반 체크** | 형식/말투 규칙을 지켰나 (예: `🐾`로 끝나나) | 자동, 해석이 쉬움 | 규칙으로 표현되는 것만 확인 가능 |
| **정답 비교 지표** | 정답과 얼마나 겹치나 (정확도, ROUGE 등) | 분류·추출처럼 정답이 정해진 과제에 좋음 | 자유로운 답변에는 잘 안 맞음 |
| **LLM 심사(LLM-as-a-judge)** | 더 큰 모델이 답변을 채점 | 자유로운 답변도 자동 평가 | 비용, 심사 모델의 편향 |
| **사람이 직접 읽기** | 전반적인 품질 | 가장 정확 | 느리고 비쌈 |
| **벤치마크** | 원래 능력을 잃지 않았나 (예: `lm-evaluation-harness`) | 표준화된 비교 | 시간이 오래 걸림 |

이 노트북에서는 앞의 세 가지(loss/perplexity, 규칙 기반 체크, 직접 읽기)와 일반 문장 loss(원래 능력 유지)를 씁니다.

### Perplexity란?

loss는 "정답 토큰의 확률에 `-log`를 씌운 평균"입니다. **perplexity = `exp(loss)`** 는 이것을 직관적인 숫자로 바꾼 것으로, "모델이 매 토큰마다 평균 몇 개의 후보 중에서 헷갈리고 있나"로 읽을 수 있습니다.
perplexity 2는 매번 두 개 중에서 고르는 정도로 확신한다는 뜻이고, 무작위로 찍으면 어휘 크기(약 15만)가 됩니다.

## 2. 평가 함수 만들기

### 2-1. 답변 부분 loss

`Trainer.evaluate()`와 같은 값을 직접 계산합니다. 프롬프트와 패딩(`-100`)을 빼고 **답변 토큰만** 평균합니다.

In [ ]:
@torch.no_grad()
def answer_loss(model, dataset, batch_size=4):
    """답변 토큰들에 대한 평균 cross entropy"""
    model.eval()
    total, count = 0.0, 0
    for i in range(0, len(dataset), batch_size):
        batch = collate([dataset[j] for j in range(i, min(i + batch_size, len(dataset)))])
        batch = {k: v.to(model.device) for k, v in batch.items()}
        logits = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits
        logits = logits[:, :-1, :].float()          # 마지막 위치는 예측할 다음 토큰이 없음
        targets = batch["labels"][:, 1:]            # 한 칸 밀어서 정답으로
        total += F.cross_entropy(
            logits.reshape(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-100, reduction="sum"
        ).item()
        count += (targets != -100).sum().item()
    return total / count


general_texts = [
    "대한민국은 동아시아의 한반도 남부에 위치한 나라이다. 수도는 서울이다.",
    "광합성은 식물이 빛 에너지를 이용해 이산화탄소와 물로 포도당을 만드는 과정이다.",
    "The quick brown fox jumps over the lazy dog.",
]


@torch.no_grad()
def text_loss(model, texts):
    """일반 문장 loss: 원래 능력을 얼마나 유지했나"""
    model.eval()
    losses = []
    for t in texts:
        batch = tokenizer(t, return_tensors="pt").to(model.device)
        losses.append(model(**batch, labels=batch["input_ids"]).loss.item())
    return sum(losses) / len(losses)

### 2-2. 답변 생성과 규칙 기반 체크

냥이 말투의 규칙을 코드로 확인합니다.

- **냥 말투:** 답변에 `냥`이 들어 있나
- **🐾로 끝남:** 답변이 `🐾`로 끝나나
- **스스로 멈춤:** `max_new_tokens`까지 가기 전에 EOS를 내고 멈췄나

평가에는 학습에 쓰지 않은 평가용 질문 6개와, 데이터에 전혀 없는 새 질문들을 함께 씁니다.

In [ ]:
@torch.no_grad()
def generate(model, question, max_new_tokens=80, **gen_kwargs):
    """답변 문자열과 '스스로 멈췄는지'를 함께 돌려준다"""
    model.eval()
    inputs = tokenizer(build_prompt(question), return_tensors="pt").to(model.device)
    settings = dict(do_sample=False, repetition_penalty=1.1)
    settings.update(gen_kwargs)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        **settings,
    )
    new_tokens = out[0, inputs["input_ids"].shape[1]:]
    stopped = tokenizer.eos_token_id in new_tokens.tolist()
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip(), stopped


def style_check(answer, stopped):
    return {"냥 말투": "냥" in answer, "🐾로 끝남": answer.endswith("🐾"), "스스로 멈춤": stopped}


eval_questions = list(raw["eval"]["question"])           # 학습에 안 쓴 평가용 질문
new_questions = [                                        # 데이터에 전혀 없는 질문
    "서울에서 부산까지 어떻게 가?",
    "LLM이 뭐야?",
    "오늘 저녁 메뉴 추천해 줘.",
    "파이썬으로 리스트를 정렬하는 방법 알려줘.",
    "잠이 안 올 때는 어떻게 해?",
]
all_questions = eval_questions + new_questions
print(f"평가 질문 {len(all_questions)}개 (평가용 {len(eval_questions)} + 새 질문 {len(new_questions)})")

## 3. 베이스 / Full / LoRA 비교

세 모델을 **하나씩** 불러와서 평가하고 메모리에서 내립니다. 세 개를 동시에 올리지 않으므로 메모리가 적어도 됩니다.

- **베이스:** `MODEL_NAME` 그대로
- **Full:** 2단계에서 저장한 모델 전체
- **LoRA:** 베이스 모델 + 3단계 어댑터

In [ ]:
def load_variant(name):
    if name == "베이스":
        model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
    elif name == "Full":
        model = AutoModelForCausalLM.from_pretrained(FULL_DIR)
    elif name == "LoRA":
        model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
        model = PeftModel.from_pretrained(model, LORA_DIR)   # 베이스 위에 어댑터를 얹음
    return model.to(device, dtype=eval_dtype).eval()


variants = ["베이스"] + (["Full"] if FULL_DIR.exists() else []) + (["LoRA"] if LORA_DIR.exists() else [])
results = {}

for name in variants:
    start = time.time()
    model = load_variant(name)
    r = {
        "eval_loss": answer_loss(model, eval_tokenized),
        "general_loss": text_loss(model, general_texts),
        "answers": {},
    }
    for q in all_questions:
        answer, stopped = generate(model, q)
        r["answers"][q] = {"answer": answer, **style_check(answer, stopped)}
    results[name] = r
    print(f"{name:6} 평가 완료 ({time.time() - start:.0f}초)")

    del model
    if device == "cuda":
        torch.cuda.empty_cache()

### 3-1. 숫자로 비교

In [ ]:
checks = ["냥 말투", "🐾로 끝남", "스스로 멈춤"]

header = f"{'':<22}" + "".join(f"{name:>12}" for name in results)
print(header)
print("-" * len(header))
print(f"{'평가 loss (답변)':<22}" + "".join(f"{r['eval_loss']:>12.3f}" for r in results.values()))
print(f"{'평가 perplexity':<22}" + "".join(f"{math.exp(r['eval_loss']):>12.1f}" for r in results.values()))
print(f"{'일반 문장 loss':<22}" + "".join(f"{r['general_loss']:>12.3f}" for r in results.values()))
for c in checks:
    rates = [sum(a[c] for a in r["answers"].values()) / len(r["answers"]) for r in results.values()]
    print(f"{c:<22}" + "".join(f"{x:>12.0%}" for x in rates))

**읽는 법**

- **평가 loss / perplexity:** 낮을수록 평가용 정답 답변을 잘 예측합니다. 베이스보다 Full과 LoRA가 크게 낮아야 합니다.
- **일반 문장 loss:** 베이스와 비슷할수록 원래 능력을 잘 유지한 것입니다. 보통 Full이 LoRA보다 조금 더 오릅니다(더 많이 잊어버림).
- **규칙 체크:** 베이스는 대부분 0%에 가깝고, 파인튜닝한 모델은 100%에 가까워야 합니다.

> 📝 평가 질문이 11개뿐이라 숫자 하나하나는 흔들릴 수 있습니다. 실무에서는 평가 데이터를 수백 개 이상 준비합니다.

### 3-2. 직접 읽어 보기

숫자만으로는 답변 **내용**이 맞는지 알 수 없습니다. 반드시 직접 읽어 보세요.

In [ ]:
for q in all_questions:
    tag = "(평가용)" if q in eval_questions else "(새 질문)"
    print(f"Q: {q} {tag}")
    for name, r in results.items():
        a = r["answers"][q]
        marks = "".join("✅" if a[c] else "❌" for c in checks)
        print(f"  [{name:4}] {marks} {a['answer']}")
    print("-" * 70)

✅/❌는 순서대로 `냥 말투`, `🐾로 끝남`, `스스로 멈춤`입니다.

볼 점:
- 말투는 둘 다 잘 배웠는데 **내용이 틀린** 답이 있나요? 0.5B는 지식이 적은 작은 모델입니다. 파인튜닝은 **말하는 방식**을 바꾸지 **지식**을 크게 늘리지는 않습니다.
- Full과 LoRA의 답변이 얼마나 비슷한가요? 이런 좁은 과제에서는 차이가 작은 것이 일반적입니다.

결과를 파일로 남겨 두면 나중에 설정을 바꿔 다시 학습했을 때 비교하기 좋습니다.

In [ ]:
REPORT_PATH = ROOT / "outputs/04_eval_results.json"
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(json.dumps(results, ensure_ascii=False, indent=2))
print("저장:", REPORT_PATH.resolve())

## 4. 생성 설정: greedy vs sampling

지금까지는 `do_sample=False`(**greedy**: 매번 확률이 가장 높은 토큰)로 생성해서 같은 질문에 항상 같은 답이 나왔습니다.
챗봇은 보통 **sampling**으로 확률에 따라 토큰을 뽑아서 다양한 답을 만듭니다.

| 설정 | 뜻 |
|---|---|
| `temperature` | 확률 분포를 날카롭게(<1) 또는 평평하게(>1). 높을수록 다양하지만 엉뚱해짐 |
| `top_p` | 누적 확률이 `p`가 될 때까지의 상위 후보 중에서만 뽑기 |
| `repetition_penalty` | 이미 나온 토큰의 확률을 깎아서 반복 억제 |

In [ ]:
best = "LoRA" if "LoRA" in variants else variants[-1]
model = load_variant(best)
q = "고양이랑 친해지려면 어떻게 해야 해?"
print(f"[{best}] Q: {q}\n")

print("greedy       :", generate(model, q)[0])
for temp in [0.7, 1.5]:
    for i in range(2):
        print(f"temp={temp} #{i + 1}  :", generate(model, q, do_sample=True, temperature=temp, top_p=0.9)[0])

del model
if device == "cuda":
    torch.cuda.empty_cache()

`temperature`가 높을수록 답이 다양해지고, 너무 높으면 말투가 무너지거나 엉뚱한 말을 하기 시작합니다. 평가는 결과가 재현되도록 greedy로, 실제 서비스는 `temperature=0.7, top_p=0.9` 정도의 sampling을 많이 씁니다.

## 5. LoRA 어댑터 합치기(merge)와 저장 정리

3단계에서 LoRA 어댑터는 베이스 모델 **위에 얹어서** 썼습니다. 이 상태로는 매 Linear마다 `W·x`와 `B·A·x`를 따로 계산하므로 추론이 조금 느립니다.

학습이 끝났다면 `W' = W + B·A × (alpha/r)`를 한 번 계산해서 **일반 모델로 합칠(merge)** 수 있습니다.

| 방식 | 저장 크기 | 불러오기 | 장점 |
|---|---|---|---|
| 어댑터로 보관 | 수십 MB | 베이스 + `PeftModel.from_pretrained` | 작음. 베이스 하나에 어댑터 여러 개를 바꿔 끼울 수 있음 |
| 병합(merge) | 베이스와 같음 (약 2GB) | `AutoModelForCausalLM.from_pretrained` 하나로 | 추론이 빠름. PEFT 없이 어디서나 사용 (vLLM, llama.cpp 변환 등) |

병합은 **fp32 베이스**에 하는 것이 안전합니다. 16비트 가중치에 작은 변화를 더하면 반올림으로 일부가 사라질 수 있기 때문입니다.

In [ ]:
if LORA_DIR.exists():
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME)               # fp32
    lora_model = PeftModel.from_pretrained(base, LORA_DIR).to(device).eval()

    # 병합 전 출력을 기록해 둔다
    check_inputs = tokenizer(build_prompt("고양이는 왜 높은 곳을 좋아해?"), return_tensors="pt").to(device)
    with torch.no_grad():
        logits_before = lora_model(**check_inputs).logits.float().cpu()

    merged = lora_model.merge_and_unload()     # W + BA를 계산해서 일반 모델로 만든다
    print("병합 후 모델 종류:", type(merged).__name__)   # PeftModel이 아닌 일반 모델

    merged.save_pretrained(MERGED_DIR)
    tokenizer.save_pretrained(MERGED_DIR)
    del lora_model, merged, base
    if device == "cuda":
        torch.cuda.empty_cache()
else:
    print("3단계 어댑터가 없어서 건너뜁니다.")

병합한 모델을 **PEFT 없이** 일반 모델로 다시 불러와서, 병합 전(베이스 + 어댑터)과 출력이 같은지 확인합니다.

In [ ]:
if MERGED_DIR.exists() and "logits_before" in globals():
    reloaded = AutoModelForCausalLM.from_pretrained(MERGED_DIR).to(device).eval()
    with torch.no_grad():
        logits_after = reloaded(**check_inputs).logits.float().cpu()

    diff = (logits_before - logits_after).abs().max().item()
    print(f"병합 전후 logits 최대 차이: {diff:.2e}  (0에 가까우면 같은 모델)")
    print("답변:", generate(reloaded, "고양이는 왜 높은 곳을 좋아해?")[0])
    del reloaded
    if device == "cuda":
        torch.cuda.empty_cache()

### 저장 크기 비교

In [ ]:
def dir_size_mb(path):
    return sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file()) / 1024**2


for label, path in [("Full (2단계)", FULL_DIR), ("LoRA 어댑터 (3단계)", LORA_DIR), ("LoRA 병합 (4단계)", MERGED_DIR)]:
    if Path(path).exists():
        print(f"{label:<20}: {dir_size_mb(path):>8.1f} MB   {path}")

> 💡 **저장/불러오기 정리**
>
> ```python
> # Full 또는 병합 모델
> model.save_pretrained(DIR); tokenizer.save_pretrained(DIR)
> model = AutoModelForCausalLM.from_pretrained(DIR)
>
> # LoRA 어댑터
> peft_model.save_pretrained(ADAPTER_DIR)
> base = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
> model = PeftModel.from_pretrained(base, ADAPTER_DIR)
>
> # 병합
> merged = model.merge_and_unload(); merged.save_pretrained(MERGED_DIR)
> ```
>
> Hugging Face Hub에 올려서 공유하려면 `huggingface-cli login`으로 로그인한 뒤 `model.push_to_hub("내아이디/cat-assistant-lora")`를 쓰면 됩니다. 어댑터만 올리면 다른 사람은 베이스 모델을 받아서 얹어 쓸 수 있습니다.

## 6. 다음 단계 ①: QLoRA로 더 큰 모델 학습하기

LoRA로도 베이스 가중치는 메모리에 올려야 합니다. bf16 기준 7B 모델은 가중치만 약 14GB라서 12GB GPU에 들어가지 않습니다.

**QLoRA**는 얼려 둔 베이스 가중치를 **4비트(NF4)로 양자화**해서 올리고, 그 위에 LoRA를 학습합니다.
계산할 때는 4비트 가중치를 bf16으로 잠깐 풀어서 쓰고, 학습하는 LoRA 가중치는 bf16/fp32로 유지합니다.

| 모델 | bf16 베이스 (LoRA) | 4비트 베이스 (QLoRA) | RTX 3060 12GB |
|---|---|---|---|
| Qwen2.5-0.5B | 약 1GB | 약 0.5GB | 둘 다 여유 |
| Qwen2.5-1.5B | 약 3GB | 약 1.2GB | 둘 다 가능 |
| Qwen2.5-3B | 약 6GB | 약 2GB | LoRA 빠듯, QLoRA 여유 |
| Qwen2.5-7B | 약 15GB | 약 5GB | **QLoRA만 가능** (batch 1, gradient checkpointing) |

(4비트 값은 임베딩 등 양자화하지 않는 부분을 포함한 대략적인 값입니다.)

**대가:** 양자화/역양자화 때문에 LoRA보다 느리고(약 30% 이상), 결과가 아주 약간 나빠질 수 있습니다. 그리고 **NVIDIA GPU가 필요**합니다(`bitsandbytes` 라이브러리).

아래 셀은 QLoRA로 모델을 불러와 LoRA를 붙이는 코드입니다. 학습 부분은 3단계와 똑같습니다(`Trainer`에 넘기기만 하면 됨).
실행하려면 `pip install bitsandbytes`로 설치하고 `RUN_QLORA = True`로 바꾸세요.

In [ ]:
RUN_QLORA = False                     # True로 바꾸면 실행 (NVIDIA GPU + bitsandbytes 필요)
QLORA_MODEL = "Qwen/Qwen2.5-1.5B"    # 12GB에서 "Qwen/Qwen2.5-3B", "Qwen/Qwen2.5-7B"도 가능

if RUN_QLORA and device == "cuda":
    from transformers import BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",              # 정규분포 가중치에 맞춘 4비트 형식
        bnb_4bit_compute_dtype=torch.bfloat16,  # 계산은 bf16으로
        bnb_4bit_use_double_quant=True,         # 양자화 상수까지 한 번 더 양자화 (조금 더 절약)
    )
    torch.cuda.reset_peak_memory_stats()
    qmodel = AutoModelForCausalLM.from_pretrained(QLORA_MODEL, quantization_config=bnb_config, device_map={"": 0})
    print(f"4비트 모델 메모리: {qmodel.get_memory_footprint() / 1024**3:.2f} GB")

    qmodel = prepare_model_for_kbit_training(qmodel)   # 학습 안정화 준비 (gradient checkpointing 포함)
    qmodel = get_peft_model(qmodel, LoraConfig(
        task_type="CAUSAL_LM", r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    ))
    qmodel.print_trainable_parameters()
    print(f"현재 VRAM 사용량: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

    # 학습은 3단계와 같습니다:
    # trainer = Trainer(model=qmodel, args=TrainingArguments(..., bf16=True,
    #                   gradient_checkpointing=True, per_device_train_batch_size=1, gradient_accumulation_steps=8),
    #                   train_dataset=..., data_collator=collate)
    # trainer.train()
    print("\n" + generate(qmodel, "고양이는 몇 시간 자?")[0])   # 아직 학습 전이라 베이스처럼 답함
elif RUN_QLORA:
    print("QLoRA(bitsandbytes 4비트)는 NVIDIA GPU가 필요합니다.")
    if device == "mps":
        print("Mac에서는 Apple의 MLX를 쓰면 4비트 모델 + LoRA 학습(QLoRA와 같은 아이디어)을 할 수 있습니다.")
        print("  pip install mlx-lm")
        print("  mlx_lm.convert --hf-path Qwen/Qwen2.5-1.5B -q            # 4비트로 변환 (mlx_model/)")
        print("  mlx_lm.lora --model mlx_model --train --data <jsonl 폴더>   # 4비트 모델 위에 LoRA 학습")
else:
    print("건너뜀. RUN_QLORA = True로 바꾸면 4비트 모델을 불러와 LoRA를 붙입니다.")

## 7. 다음 단계 ②: 나만의 SFT 데이터 만들기

이 과정에서 쓴 데이터는 42개의 질문/답변 쌍이었습니다. 실제 과제에 파인튜닝하려면 **데이터 만들기가 가장 중요하고 시간도 가장 많이 드는 일**입니다.

### 좋은 SFT 데이터의 조건

1. **품질 > 양:** 잘 만든 수백~수천 개가 대충 만든 수만 개보다 낫습니다. 모델은 데이터의 실수까지 그대로 배웁니다.
2. **일관된 형식:** 말투, 길이, 구조가 원하는 출력과 정확히 같아야 합니다.
3. **다양한 입력:** 질문의 주제, 길이, 표현 방식을 다양하게. 비슷한 질문만 있으면 그 밖의 질문에서 무너집니다.
4. **평가 데이터는 따로:** 학습에 쓰지 않은 예시로 평가해야 "외웠는지, 배웠는지"를 알 수 있습니다. 중복이 섞이지 않게 주의합니다.
5. **출처와 라이선스 확인:** 다른 LLM으로 생성한 데이터는 그 모델의 이용 약관을 확인하세요.

학습 전에 데이터를 간단히 점검하는 코드를 만들어 봅시다. 실제로 데이터를 만들 때 이런 점검을 꼭 해 보세요.

In [ ]:
def check_dataset(path, eval_path=None):
    rows = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
    questions = [r["question"].strip() for r in rows]
    answers = [r["answer"].strip() for r in rows]
    lengths = [len(tokenizer(build_prompt(q) + a)["input_ids"]) for q, a in zip(questions, answers)]

    print(f"예시 수: {len(rows)}")
    print(f"토큰 길이: 최소 {min(lengths)}, 평균 {sum(lengths) / len(lengths):.0f}, 최대 {max(lengths)}")
    print(f"중복 질문: {len(questions) - len(set(questions))}개")
    print(f"빈 답변: {sum(1 for a in answers if not a)}개")

    bad = [a for a in answers if not (a.endswith("🐾") and "냥" in a)]
    print(f"말투 규칙을 어긴 답변: {len(bad)}개")
    for a in bad[:3]:
        print("   ", a)

    if eval_path:
        eval_q = {json.loads(line)["question"].strip() for line in open(eval_path, encoding="utf-8") if line.strip()}
        leaked = eval_q & set(questions)
        print(f"평가 데이터와 겹치는 질문: {len(leaked)}개 {'⚠️' if leaked else '✅'}")


check_dataset(DATA_DIR / "cat_assistant_train.jsonl", DATA_DIR / "cat_assistant_eval.jsonl")

### 실무에서는 chat template을 씁니다

이 과정에서는 원리를 보기 위해 `### 질문:` / `### 답변:` 형식을 직접 정했습니다.
실제로는 **대화 튜닝된 Instruct 모델**(예: `Qwen/Qwen2.5-0.5B-Instruct`)에서 시작해서, 그 모델의 **chat template** 형식으로 학습하는 경우가 많습니다.
데이터는 보통 이런 `messages` 형식으로 저장합니다.

```json
{"messages": [
  {"role": "system", "content": "너는 고양이 비서 냥이야."},
  {"role": "user", "content": "한국의 수도는 어디야?"},
  {"role": "assistant", "content": "한국의 수도는 서울이다냥! 🐾"}
]}
```

`tokenizer.apply_chat_template()`이 이것을 모델 고유의 형식으로 바꿔 줍니다. 우리 데이터를 바꿔서 확인해 봅시다.

In [ ]:
def to_messages(example, system_prompt="너는 친절한 고양이 비서 냥이야."):
    return {"messages": [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": example["question"]},
        {"role": "assistant", "content": example["answer"]},
    ]}


chat_example = to_messages(raw["train"][0])
print(json.dumps(chat_example, ensure_ascii=False, indent=1))

if tokenizer.chat_template:
    print("\n--- apply_chat_template 결과 ---")
    print(tokenizer.apply_chat_template(chat_example["messages"], tokenize=False))
else:
    print("\n이 토크나이저에는 chat template이 없습니다. Instruct 모델의 토크나이저를 쓰세요.")

`<|im_start|>`, `<|im_end|>` 같은 특수 토큰이 역할을 구분합니다. 학습 시 **assistant 부분만 loss를 계산**하는 원리는 2단계에서 한 `-100` 마스킹과 같습니다.

직접 마스킹 코드를 짜는 대신 **TRL** 라이브러리의 `SFTTrainer`를 쓰면 `messages` 형식 데이터를 받아 chat template 적용, 마스킹, LoRA 연결까지 해 줍니다.
이 과정에서 `Trainer`와 collator를 직접 짠 이유는 `SFTTrainer`가 안에서 무슨 일을 하는지 이해하기 위해서였습니다.

### 데이터를 모으는 방법

| 방법 | 설명 |
|---|---|
| 직접 작성 | 가장 품질이 높음. 처음 수십~수백 개는 직접 쓰는 것을 추천 |
| 기존 로그/문서 변환 | 고객 문의 기록, FAQ, 매뉴얼을 질문/답변 쌍으로 정리 |
| 큰 모델로 생성 후 검수 | 직접 쓴 예시 몇 개를 보여 주고 비슷한 예시를 생성하게 한 뒤 사람이 검수 |
| 공개 데이터셋 | Hugging Face Hub에서 과제에 맞는 데이터셋을 찾아 일부 사용 (라이선스 확인) |

## 8. 그 다음에 배울 것들

| 주제 | 내용 |
|---|---|
| **선호도 학습 (DPO, RLHF)** | "좋은 답 vs 나쁜 답" 쌍으로 모델의 선호를 조정. SFT 다음 단계의 post-training. 이 과정의 5단계에서 다룹니다 |
| **TRL `SFTTrainer`** | 실무용 SFT 도구. chat template, 마스킹, 패킹(packing), LoRA 연결을 자동으로 |
| **더 큰 데이터 / 더 큰 모델** | 수천 개 데이터 + 1.5B~7B 모델을 QLoRA로 |
| **체계적인 평가** | `lm-evaluation-harness` 벤치마크, LLM-as-a-judge |
| **배포** | 병합 모델을 vLLM으로 서빙하거나, GGUF로 변환해서 llama.cpp / Ollama로 CPU에서 실행 |
| **분산 학습** | 여러 GPU로 학습: DeepSpeed, FSDP (`accelerate`로 설정) |

## 9. 전체 과정 정리

| 단계 | 핵심 |
|---|---|
| 1. 기초 | LLM 학습 = 다음 토큰 예측 + cross entropy. 프리트레이닝은 대량 텍스트, 파인튜닝은 작은 목적 데이터. Full FT 메모리 ≈ 파라미터 × 16바이트 |
| 2. Full 파인튜닝 | HF `Trainer`, 답변만 loss(`-100`), EOS, 혼합 정밀도, gradient accumulation. 모든 가중치 학습, 저장도 모델 전체 |
| 3. LoRA | 원래 가중치는 얼리고 작은 `A`, `B`만 학습. 메모리와 저장 크기가 크게 줄고 결과는 비슷. 12GB에서 1.5B도 가능 |
| 4. 평가와 정리 | loss/perplexity + 규칙 체크 + 직접 읽기로 평가. 어댑터 병합. 더 큰 모델은 QLoRA, 결과의 품질은 데이터가 결정 |

### 연습 문제

1. 2단계나 3단계에서 `epochs`를 바꿔 다시 학습한 뒤 이 노트북을 다시 실행해서, 평가 표가 어떻게 변하는지 보세요. (`outputs/04_eval_results.json`을 이전 결과와 비교)
2. `style_check`에 나만의 규칙(예: 답변이 두 문장 이하인가)을 추가해 보세요.
3. `new_questions`에 일부러 어려운 질문(긴 질문, 영어 질문, 코드 질문)을 넣어 보세요. 어떤 질문에서 말투가 무너지나요?
4. 나만의 말투/규칙을 가진 데이터를 50개 만들고 `check_dataset`으로 점검한 뒤, 3단계 LoRA로 학습해 보세요.
5. (GPU) `RUN_QLORA = True`로 `Qwen/Qwen2.5-3B`를 불러와서 3단계와 같은 방식으로 학습해 보세요.

수고하셨습니다! 🐾